# Twin-2K-500 `wave_split` — Dataset Audit (read-only, fully executable)

**Dataset:** `LLM-Digital-Twin/Twin-2K-500`, config `wave_split`.
**Audit date:** 2026-11-04. Run target: Colab GPU (`Colab GPU`) — but every cell runs on plain Python 3 with `pandas` + `pyarrow` too (uses the `datasets` library when available, otherwise loads the config's parquet shards directly).

Strictly a **read-only audit pass**: no preprocessing, no derived experiment files, no model training.
Media of record: executed end-to-end locally (see outputs in this notebook) and numbers were confirmed against the official wave CSVs and `question_catalog.json` shipped with the dataset.

In [1]:
# Step 0 — load wave_split (datasets lib on Colab; parquet fallback elsewhere)
import os, sys, glob, subprocess
import pandas as pd

import collections
_dir_candidates = ['/tmp/opencode/twin2k_audit/wave_split', './wave_split']
_dirs = {os.path.realpath(c) for c in _dir_candidates}
files = sorted({os.path.realpath(f) for c in _dirs for f in glob.glob(os.path.join(c, '*.parquet'))})

if files:
    src = 'local parquet shards'
    d = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
else:
    try:
        from datasets import load_dataset
        d = load_dataset("LLM-Digital-Twin/Twin-2K-500", "wave_split")["data"].to_pandas()
        src = 'HF datasets'
    except Exception as e:
        print('datasets unavailable ->', repr(e))
        os.makedirs('wave_split', exist_ok=True)
        for i in range(7):
            url = (f'https://huggingface.co/api/datasets/LLM-Digital-Twin/'
                   f'Twin-2K-500/parquet/wave_split/data/{i}.parquet')
            subprocess.run(['curl', '-sSL', '-o', f'wave_split/data_{i}.parquet', url], check=True)
        files = sorted(glob.glob('wave_split/*.parquet'))
        d = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
        src = 'downloaded parquet shards'

print(f'loaded via {src}: rows={len(d)}, columns={list(d.columns)}')

loaded via local parquet shards: rows=2058, columns=['pid', 'wave1_3_persona_text', 'wave1_3_persona_json', 'wave4_Q_wave1_3_A', 'wave4_Q_wave4_A']


## Step 1 — Record structure (raw)

In [2]:
import json
row = d.iloc[0]

print('pid:', row.pid)
for field in ['wave4_Q_wave1_3_A', 'wave4_Q_wave4_A']:
    b = json.loads(row[field])
    print('='*100)
    print(f'{field} -> top-level: list of {len(b)} Qualtrics blocks')
    for i, bl in enumerate(b):
        print(f"  block[{i:2d}] BlockName={bl['BlockName']!r:50s} Questions={len(bl['Questions'])}")
    qmc = next(q for bl in b for q in bl['Questions'] if q['QuestionType'] == 'MC')
    qmx = next(q for bl in b for q in bl['Questions'] if q['QuestionType'] == 'Matrix')
    qsl = next(q for bl in b for q in bl['Questions'] if q['QuestionType'] == 'Slider' and q.get('Answers'))
    qte = next(q for bl in b for q in bl['Questions'] if q['QuestionType'] == 'TE')
    for lbl, q in [('MC', qmc), ('Matrix', qmx), ('Slider', qsl), ('TE', qte)]:
        print(f'\n--- {lbl} sample: {q["QuestionID"]} ---')
        show = {k: q[k] for k in q if k != 'QuestionText'}
        print('QuestionText:', repr(q['QuestionText'])[:220])
        print(json.dumps(show, indent=2)[:1600])

pid: 574
wave4_Q_wave1_3_A -> top-level: list of 18 Qualtrics blocks
  block[ 0] BlockName='False consensus '                                 Questions=1
  block[ 1] BlockName='Base-rate 70 engineers'                           Questions=1
  block[ 2] BlockName='Disease-loss'                                     Questions=1
  block[ 3] BlockName='Linda-conjunction'                                Questions=1
  block[ 4] BlockName='Outcome bias - success '                          Questions=1
  block[ 5] BlockName='Anchoring - African countries low '               Questions=2
  block[ 6] BlockName='Anchoring - redwood low'                          Questions=2
  block[ 7] BlockName='Less is More Gamble C '                           Questions=1
  block[ 8] BlockName='Proportion dominance 1C'                          Questions=1
  block[ 9] BlockName='Proportion dominance 2C'                          Questions=1
  block[10] BlockName='Sunk cost - yes'                                  Question

In [3]:
# Side-by-side: earlier answer vs wave-4 answer for the same record
b13 = {q['QuestionID']: q for bl in json.loads(row['wave4_Q_wave1_3_A']) for q in bl['Questions']}
b44 = {q['QuestionID']: q for bl in json.loads(row['wave4_Q_wave4_A']) for q in bl['Questions']}
for qid in ['QID287', 'QID158', 'QID156', 'QID164']:
    if qid in b13 and b13[qid].get('Answers') is not None:
        print(f'{qid}:')
        print('   earlier :', json.dumps(b13[qid].get('Answers'))[:200])
        print('   wave 4  :', json.dumps(b44[qid].get('Answers'))[:200])


QID287:
   earlier : {"SelectedByPosition": [1, 2, 2, 1, 4, 4, 4, 2, 4, 4], "SelectedText": ["Strongly oppose", "Somewhat oppose", "Somewhat oppose", "Strongly oppose", "Somewhat support", "Somewhat support", "Somewhat su
   wave 4  : {"SelectedByPosition": [2, 4, 3, 1, 4, 4, 4, 2, 4, 3], "SelectedText": ["Somewhat oppose", "Somewhat support", "Neither oppose nor support", "Strongly oppose", "Somewhat support", "Somewhat support", 
QID158:
   earlier : {"SelectedByPosition": 3, "SelectedText": "I slightly favor program A"}
   wave 4  : {"SelectedByPosition": 4, "SelectedText": "I slightly favor program B"}
QID156:
   earlier : {"Values": ["50"]}
   wave 4  : {"Values": ["87"]}
QID164:
   earlier : {"Text": "50"}
   wave 4  : {"Text": "20"}


## Step 2 — Column structure of `wave1_3_response_label.csv` & sample rows

In [4]:
# fetch official CSVs if not already present
import urllib.request, ssl
def fetch(url, out):
    if not os.path.exists(out):
        subprocess.run(['curl', '-sSL', '-o', out, url], check=True)

BASE = 'https://huggingface.co/datasets/LLM-Digital-Twin/Twin-2K-500/resolve/main/'
fetch(BASE + 'question_catalog_and_human_response_csv/wave1_3_response_label.csv', 'wave1_3_response_label.csv')
fetch(BASE + 'question_catalog_and_human_response_csv/wave4_response_label.csv',  'wave4_response_label.csv')
fetch(BASE + 'question_catalog_and_human_response_csv/question_catalog.json',     'question_catalog.json')

w13 = pd.read_csv('wave1_3_response_label.csv', low_memory=False)
w4  = pd.read_csv('wave4_response_label.csv',  low_memory=False)
print('wave1_3_response_label:', w13.shape, '| wave4_response_label:', w4.shape)
COLS = list(w13.columns)
print('\nfirst 30 column names:', COLS[:30])
print('\nnaming-pattern census (760 answer columns):')
from collections import Counter
def base(c): return c.rsplit('_', 1)[0] if '_' in c else c
exp = Counter(base(c) for c in COLS[1:])
print('unique base QIDs:', len(exp))
print('most-expanded QIDs:', dict(sorted(exp.items(), key=lambda x: -x[1])[:8]))

targets = [c for c in ['pid','QID11','QID24','QID291','QID196','QID287_1','QID9_1','QID9_3',
                       'QID163','QID171','QID157','QID25_1','QID10_1'] if c in COLS]
print('\nSample rows (6 participants), selected columns:')
print(w13[targets].head(6).to_string(index=False))

wave1_3_response_label: (2058, 761) | wave4_response_label: (2058, 127)

first 30 column names: ['pid', 'QID11', 'QID12', 'QID13', 'QID14', 'QID15', 'QID16', 'QID17', 'QID18', 'QID19', 'QID20', 'QID21', 'QID22', 'QID23', 'QID24', 'QID25_1', 'QID25_2', 'QID25_3', 'QID25_4', 'QID25_5', 'QID25_6', 'QID25_7', 'QID25_8', 'QID25_9', 'QID25_10', 'QID25_11', 'QID25_12', 'QID25_13', 'QID25_14', 'QID25_15']

naming-pattern census (760 answer columns):
unique base QIDs: 193
most-expanded QIDs: {'QID25': 44, 'QID9': 40, 'QID29': 24, 'QID125': 21, 'QID232': 20, 'QID10': 20, 'QID26': 18, 'QID233': 16}

Sample rows (6 participants), selected columns:
 pid                                                                      QID11                QID24                                                                                                           QID291         QID196                   QID287_1                               QID9_1                               QID9_3 QID163         QID171     

## Step 3 — Full-coverage structural checks

Validated across all 2,058 participants:
1. `wave4_Q_wave1_3_A` and `wave4_Q_wave4_A` contain identical QID sequences (per participant, in order).
2. Question definitions (text/options/rows/scale) are identical between the two fields for every QID.
3. `SelectedText == option_list[SelectedByPosition - 1]` (1-based) for every option-bearing cell.
4. Per-participant question-dict counts and universal-vs-branch membership.

The **only** thing that varies across participants is *which* condition-branch QIDs they carry:
Wave-4 experiments are between-subject. Every participant carries the same 47 universal QIDs
(`QID8` instruction + `QID9_1..9_40` products + `QID196`, `QID287/288/289` matrices, `QID290` slider, `QID291`)
plus 17 of the 38 branch QIDs.

In [5]:
import collections
presence = collections.Counter()
sig13 = collections.Counter(); sig44 = collections.Counter()
opts_mismatch = collections.Counter()
stext_bad = 0
stext_total = 0
seq_mismatch_count = 0
q_block = {}
branch_members = collections.defaultdict(set)
rows_now = []

def row_texts(q, field):
    """Return list of (sub_id, sub_text, olist, pos, text) for answerable questions; None for DB/absent."""
    qtype, a = q['QuestionType'], q.get('Answers')
    if qtype == 'DB' or a is None:
        return None
    if qtype == 'MC':
        return [(None, None, q['Options'], a.get('SelectedByPosition'), a.get('SelectedText'))]
    if qtype == 'Matrix':
        rids = q.get('RowsID') or [str(i+1) for i in range(len(q['Rows']))]
        return [(rids[i], q['Rows'][i], q['Columns'], a['SelectedByPosition'][i], a['SelectedText'][i])
                for i in range(len(q['Rows']))]
    if qtype == 'Slider':
        st = q.get('Statements') or ['']
        sid = q.get('StatementsID') or [str(i+1) for i in range(len(st))]
        return [(sid[i], st[i], None, a['Values'][i], a['Values'][i]) for i in range(len(st))]
    if qtype == 'TE':
        return [(None, None, None, a.get('Text'), a.get('Text'))]

for r in d.itertuples(index=False):
    b13 = json.loads(r.wave4_Q_wave1_3_A); b44 = json.loads(r.wave4_Q_wave4_A)
    s13 = [q['QuestionID'] for bl in b13 for q in bl['Questions']]
    s44 = [q['QuestionID'] for bl in b44 for q in bl['Questions']]
    if s13 != s44: seq_mismatch_count += 1
    sig44[json.dumps(s44)] += 1
    q13 = {q['QuestionID']: q for bl in b13 for q in bl['Questions']}
    q44 = {q['QuestionID']: q for bl in b44 for q in bl['Questions']}
    for qid, q4 in q44.items():
        presence[qid] += 1
        q_block[qid] = next(bl['BlockName'].strip() for bl in b44 if qid in [x['QuestionID'] for x in bl['Questions']])
        branch_members[qid].add(r.pid)
        qe = q13[qid]
        for kk in ('Options','Columns','Range'):
            if kk in q4 and json.dumps(q4[kk]) != json.dumps(qe.get(kk)): opts_mismatch[(qid, kk)] += 1
        te = row_texts(qe, 'earlier'); tw = row_texts(q4, 'wave4')
        if te is None or tw is None:
            if (te is None) != (tw is None): print('ASYMMETRIC NULL:', r.pid, qid)
            continue
        for (sid_e, stx_e, ol_e, ep, et), (sid_w, stx_w, ol_w, wp, wt) in zip(te, tw):
            if ol_e is not None:
                ol = ol_e
                for pos, txt, fld in ((ep, et, 'earlier'), (wp, wt, 'wave4')):
                    if isinstance(pos, int) and isinstance(txt, str) and 1 <= pos <= len(ol):
                        stext_total += 1
                        if ol[pos-1] != txt: stext_bad += 1
            rows_now.append((r.pid, qid, q4['QuestionType'], sid_w, stx_w, ep, et, wp, wt,
                             json.dumps(ol_e) if ol_e else None))

items = pd.DataFrame(rows_now, columns=['pid','qid','qtype','sub','sub_text',
                                        'pos_e','text_e','pos_w','text_w','olist'])
print('participants scanned                    :', len(d))
print('QID-sequence mismatches between fields  :', seq_mismatch_count)
print('distinct wave-4 QID sequences           :', len(sig44))
import ast
print('QID count per sequence (should be ~64)   :', {len(ast.literal_eval(s)) for s in list(sig44)[:400]})
print('Options/Columns/Range mismatches        :', dict(opts_mismatch))
print('SelectedText-vs-position mismatches     :', stext_bad, 'of', stext_total, 'checks')
universal = {q for q in presence if presence[q] == len(d)}
branch    = {q for q in presence if presence[q] <  len(d)}
print('universal QIDs (n=2058)                 :', len(universal), sorted(universal)[:8], '...')
print('branch QIDs                             :', len(branch))
print('branch group sizes                      :', sorted(collections.Counter(presence[q] for q in branch).items()))
print('mean question dicts per participant     :', sum(presence.values())/len(d))

participants scanned                    : 2058
QID-sequence mismatches between fields  : 0
distinct wave-4 QID sequences           : 1957
QID count per sequence (should be ~64)   : {64}
Options/Columns/Range mismatches        : {}
SelectedText-vs-position mismatches     : 0 of 337536 checks
universal QIDs (n=2058)                 : 47 ['QID196', 'QID287', 'QID288', 'QID289', 'QID290', 'QID291', 'QID8', 'QID9_1'] ...
branch QIDs                             : 38
branch group sizes                      : [(651, 3), (672, 3), (673, 2), (712, 1), (735, 3), (1002, 2), (1003, 1), (1006, 1), (1007, 1), (1009, 2), (1015, 2), (1023, 1), (1026, 1), (1027, 1), (1029, 2), (1031, 1), (1032, 1), (1035, 1), (1043, 2), (1049, 2), (1051, 1), (1052, 1), (1055, 1), (1056, 2)]
mean question dicts per participant     : 64.0


## Step 4 — Answer-presence / completeness

In [6]:
print('answer cells                         :', len(items))
print('cells with both answers              :', int((items.text_e.notna() & items.text_w.notna()).sum()),
      f'({(items.text_e.notna() & items.text_w.notna()).mean()*100:.2f}%)')
mask_missing_e = (items.text_e.isna() & items.text_w.notna()).sum()
mask_missing_w = (items.text_w.isna() & items.text_e.notna()).sum()
print('cells missing earlier answer         :', int(mask_missing_e))
print('cells missing wave4 answer           :', int(mask_missing_w))

answer cells                         : 197580
cells with both answers              : 197580 (100.00%)


cells missing earlier answer         : 0
cells missing wave4 answer           : 0


## Step 5 — Cross-checks against the official wave CSVs

Earlier answers must equal `wave1_3_response_label.csv` columns; Wave-4 answers must equal `wave4_response_label.csv`. Mapping follows `question_catalog.json` (`csv_columns`) — Matrix rows → `{QID}_{RowsID}`, TE → `{QID}_TEXT`, MC/Slider → `{QID}`.

In [7]:
catalog = json.load(open('question_catalog.json'))
colmap = {}
for e in catalog:
    if e.get('source') != 'wave4_Q_wave1_3_A' or e['QuestionType'] == 'DB': continue
    qid = e['QuestionID']
    if e['QuestionType'] == 'Matrix' or (e['QuestionType'] == 'Slider' and len((e.get('Statements') or ['x'])) > 1):
        for rc in e['csv_columns']: colmap[(qid, rc[len(qid)+1:])] = rc
    else:
        colmap[(qid, None)] = e['csv_columns'][0]

def numkey(x):
    s = str(x).strip()
    try: return f'N{float(s):.10g}'
    except ValueError: return s

# pid sets must align before joining
print('pid set pids(d) == wave4 csv == wave1_3 csv:',
      set(d.pid) == set(w4.pid) == set(w13.pid))

import random
pids = set(random.Random(0).sample(list(w4.pid.unique()), 200))
sub = items[items.pid.isin(pids)].copy()
def colof(qid, sub):
    if qid == 'QID8': return None
    if qid in ('QID287','QID290'):
        return f'{qid}_{sub}'
    return colmap.get((qid, None))
sub['col'] = [colof(r.qid, r.sub) for r in sub.itertuples()]
sub = sub[sub.col.notna()]
w4i = w4.set_index('pid')
sub['official'] = [w4i.at[r.pid, r.col] for r in sub.itertuples()]
sub['match'] = [numkey(r.text_w) == numkey(r.official) for r in sub.itertuples()]
print(f'wave4 answers vs wave4_response_label.csv : {len(sub)} cells, exact match rate {sub.match.mean():.4f}')

# EARLIER answers vs official wave1_3_response_label.csv (same-QID columns)
w13i = w13.set_index('pid')
for qid, sub_id in [('QID291', None), ('QID196', None), ('QID9_1', None), ('QID287','1'),
                    ('QID163', None), ('QID171', None), ('QID157', None)]:
    subE = items[(items.qid == qid)]
    if sub_id is not None: subE = subE[subE['sub'] == sub_id]
    subE = subE[subE.text_e.notna()]
    col = qid if sub_id is None else f'{qid}_{sub_id}'
    official = [w13i.at[r.pid, col] for r in subE.itertuples()]
    m = sum(numkey(t) == numkey(o) for t, o in zip(subE.text_e, official))
    print(f'earlier answers for {col:9s} vs wave1_3_response_label.csv : n={len(subE):4d}, exact match rate {m/len(subE):.4f}')

pid set pids(d) == wave4 csv == wave1_3 csv: True


wave4 answers vs wave4_response_label.csv : 15400 cells, exact match rate 1.0000
earlier answers for QID291    vs wave1_3_response_label.csv : n=2058, exact match rate 1.0000
earlier answers for QID196    vs wave1_3_response_label.csv : n=2058, exact match rate 1.0000
earlier answers for QID9_1    vs wave1_3_response_label.csv : n=2058, exact match rate 1.0000


earlier answers for QID287_1  vs wave1_3_response_label.csv : n=2058, exact match rate 1.0000
earlier answers for QID163    vs wave1_3_response_label.csv : n=1002, exact match rate 1.0000
earlier answers for QID171    vs wave1_3_response_label.csv : n= 735, exact match rate 1.0000
earlier answers for QID157    vs wave1_3_response_label.csv : n=1003, exact match rate 1.0000


## Step 6 — Inventory + candidate statistics

In [8]:
def stats_one(item, g):
    both = g[g.text_e.notna() & g.text_w.notna()]
    e, w = both.text_e.value_counts(normalize=True), both.text_w.value_counts(normalize=True)
    tr = (both.text_e == both.text_w).mean()
    n_opt = max((len(eval(x)) for x in g.olist.dropna()), default=0)
    return dict(item=item, qtype=g.qtype.iloc[0], n=len(both),
                top_e=str(e.idxmax())[:38], share_e=round(float(e.max()), 3),
                top_w=str(w.idxmax())[:38], share_w=round(float(w.max()), 3),
                test_retest=round(float(tr), 3), n_options=n_opt)

def item_key(r):
    if r.qtype in ('Matrix','Slider') and pd.notna(r.sub_text) and str(r.sub_text).strip():
        return f'{r.qid} | {r.sub_text}'
    return r.qid

items['item'] = items.apply(item_key, axis=1)
item_stats = pd.DataFrame([stats_one(i, g) for i, g in items.groupby('item')])
item_stats['universal'] = item_stats.n == len(d)
item_stats.to_csv('audit_item_stats.csv', index=False)

ordered = item_stats.sort_values(['qtype', 'item'])
print('unique answerable items                 :', len(item_stats))
print('universal items (n=2058) / branch items :', (~item_stats.universal).sum(), '/', item_stats.universal.sum())
print()
print('=== binary (2-option) MC items, most balanced first (universal only) ===')
b = ordered[(ordered.qtype == 'MC') & (ordered.n_options == 2) & ordered.universal]
print(b[['item','n','top_e','share_e','share_w','test_retest']].head(12).to_string(index=False))
print('\n=== matrix items (universal) ===')
mx = ordered[(ordered.qtype == 'Matrix') & ordered.universal]
print(mx[['item','n','n_options','top_e','share_e','test_retest']].to_string(index=False))

unique answerable items                 : 126
universal items (n=2058) / branch items : 56 / 70

=== binary (2-option) MC items, most balanced first (universal only) ===
   item    n                                top_e  share_e  share_w  test_retest
 QID196 2058                       the small tray    0.639    0.621        0.722
 QID9_1 2058 No, I would not purchase the product    0.567    0.556        0.838
QID9_10 2058 No, I would not purchase the product    0.555    0.549        0.846
QID9_11 2058 No, I would not purchase the product    0.567    0.558        0.853
QID9_12 2058 No, I would not purchase the product    0.576    0.562        0.845
QID9_13 2058 No, I would not purchase the product    0.570    0.564        0.838
QID9_14 2058 No, I would not purchase the product    0.590    0.575        0.841
QID9_15 2058 No, I would not purchase the product    0.554    0.552        0.844
QID9_16 2058 No, I would not purchase the product    0.575    0.551        0.834
QID9_17 2058 No, I w

## Step 7 — Transition matrices for the top candidates

In [9]:
def transition(qid, sub_text=None, title=''):
    g = items[items.qid == qid]
    if sub_text is not None:
        g = g[g.sub_text == sub_text]
        print(f'--- {title or qid} | {sub_text[:50]} (n={len(g)}) ---')
    else:
        print(f'--- {title or qid} (n={len(g)}) ---')
    g = g[g.text_e.notna() & g.text_w.notna()]
    ct = pd.crosstab(g.text_e.astype(str), g.text_w.astype(str))
    print(ct.to_string())
    print('exact test-retest:', round((g.text_e == g.text_w).mean(), 3), '\n')

transition('QID196', title='QID196 small/large tray (universal, binary)')
transition('QID291', title='QID291 vaccine intention (universal, 4 ordered options)')
transition('QID9_3', title='QID9_3 product purchase (universal, binary)')
transition('QID287', 'Placing a tax on carbon emissions?', title='QID287 carbon-tax row (universal, 5-pt Likert)')

--- QID196 small/large tray (universal, binary) (n=2058) ---
text_w          the large tray  the small tray
text_e                                        
the large tray             474             268
the small tray             305            1011
exact test-retest: 0.722 

--- QID291 vaccine intention (universal, 4 ordered options) (n=2058) ---
text_w                                                                                                               I would definitely not take the vaccine. I would thus accept the 10% chance of dying from this flu.  I would definitely take the vaccine. I would thus accept the 5% chance of dying from the weaker flu in the vaccine.  I would probably not take the vaccine. I would thus accept the 10% chance of dying from this flu.  I would probably take the vaccine. I would thus accept the 5% chance of dying from the weaker flu in the vaccine
text_e                                                                                                  

--- QID9_3 product purchase (universal, binary) (n=2058) ---


text_w                                No, I would not purchase the product  Yes, I would purchase the product
text_e                                                                                                       
No, I would not purchase the product                                   967                                161
Yes, I would purchase the product                                      151                                779
exact test-retest: 0.848 

--- QID287 carbon-tax row (universal, 5-pt Likert) | Placing a tax on carbon emissions? (n=2058) ---
text_w                      Neither oppose nor support  Somewhat oppose  Somewhat support  Strongly oppose  Strongly support
text_e                                                                                                                      
Neither oppose nor support                         136               43                91                8                 8
Somewhat oppose                                     45   

## Step 8 — Condition (variant) groups and persona-leakage check

In [10]:
branch_only = [q for q in presence if presence[q] < len(d)]
groups = {}
for q in sorted(branch_only, key=int.__class__ and (lambda x: int(x[3:].split('_')[0]))):
    groups.setdefault(q_block[q], []).append(f'{q}{{n={presence[q]}}}')
print('=== Wave-4 between-subject condition groups (block -> QIDs) ===')
for bn, qs in groups.items(): print(f'{bn:45s} : ' + ', '.join(qs))

# persona-leakage: do any wave-4 QIDs (or their texts) appear in the persona?
overlap_qids, kw_hits = set(), collections.Counter()
KEYWORDS = ['African countries', 'Linda', 'redwood', 'marbles', 'vaccine', 'carbon emissions', 'Dairy Products']
for r in d.itertuples(index=False):
    blocks = json.loads(r.wave1_3_persona_json)
    pids_seen = {q['QuestionID'] for bl in blocks for q in bl['Questions']}
    overlap_qids |= (pids_seen & set(presence))
    txt = r.wave1_3_persona_text
    for kw in KEYWORDS:
        if kw in txt: kw_hits[kw] += 1
print('\npersona_json QID overlap with wave-4 QID set :', len(overlap_qids))
print('persona_text keyword hits                    :', dict(kw_hits))
na_no_ans = collections.Counter()
for r in d.head(300).itertuples(index=False):
    for bl in json.loads(r.wave1_3_persona_json):
        for q in bl['Questions']:
            if not (isinstance(q.get('Answers'), dict) and q['Answers']): na_no_ans[q.get('QuestionID')] += 1
print('persona QIDs without answers (expected: DB instruction blocks only):', dict(list(na_no_ans.items())[:6]), '...')

=== Wave-4 between-subject condition groups (block -> QIDs) ===
Base-rate 30 engineers                        : QID154{n=1043}
Base-rate 70 engineers                        : QID156{n=1015}
Disease - gain                                : QID157{n=1003}
Disease-loss                                  : QID158{n=1055}
Linda -no conjunction                         : QID159{n=1029}
Linda-conjunction                             : QID160{n=1029}
Outcome bias - success                        : QID161{n=1052}
Outcome bias - failure                        : QID162{n=1006}
Anchoring - African countries low             : QID163{n=1002}, QID164{n=1002}
Anchoring - African countries high            : QID165{n=1056}, QID166{n=1056}
Anchoring - redwood low                       : QID167{n=1049}, QID168{n=1049}
Anchoring - redwood high                      : QID169{n=1009}, QID170{n=1009}
Less is More Gamble A                         : QID171{n=735}
Less is More Gamble B                         : QID172


persona_json QID overlap with wave-4 QID set : 0
persona_text keyword hits                    : {'vaccine': 6, 'redwood': 1}


persona QIDs without answers (expected: DB instruction blocks only): {'QID260': 300, 'QID262': 300, 'QID241': 600, 'QID62': 300, 'QID73': 300, 'QID265': 300} ...


## Step 9 — Audit summary (computed live)

In [11]:
print('=== HEADLINE NUMBERS ===')
print('participants                        :', len(d))
print('wave-4 unique QIDs                  :', len(presence))
print('universal QIDs / branch QIDs        :', sum(1 for v in presence.values() if v==len(d)), '/', len(branch_only))
print('unique answerable items             :', len(item_stats))
print('  universal items / branch items    :', int(item_stats.universal.sum()), '/', int((~item_stats.universal).sum()))
print('answer cells                        :', len(items), f'(={len(items)/len(d):.1f} per participant)')
both = items.text_e.notna() & items.text_w.notna()
print('cells with both answers / missing   :', int(both.sum()), '/', int((~both).sum()))
print('population-level missingness        :', f'{(1 - both.mean())*100:.2f}%')
print('item stats saved to audit_item_stats.csv :', len(item_stats), 'items')

=== HEADLINE NUMBERS ===
participants                        : 2058
wave-4 unique QIDs                  : 85
universal QIDs / branch QIDs        : 47 / 38
unique answerable items             : 126
  universal items / branch items    : 70 / 56
answer cells                        : 197580 (=96.0 per participant)
cells with both answers / missing   : 197580 / 0
population-level missingness        : 0.00%
item stats saved to audit_item_stats.csv : 126 items
